# Phase 4: Production ML Deployment & MLOps Infrastructure

Welcome to the final phase of the AuraCart MLOps initiative. In this notebook, we bridge the gap between offline experimentation and production inference. We will programmatically interact with Google Cloud Platform (GCP) to register our champion model, containerize it using Vertex AI's pre-built environments, and instantiate a live RESTful API endpoint.

As mandated by the AuraCart mandate, this system provides a unified, scalable interface for frontend microservices to query predictive logic in real-time using JSON payloads.

In [ ]:
import os
import joblib
import pandas as pd
from google.cloud import storage
from google.cloud import aiplatform

# --- Configuration Variables (Production Parameters) ---
PROJECT_ID = "dilsara"  # RESOLVED: Using reference project ID
REGION = "asia-southeast1" # RESOLVED: Using reference region
BUCKET_NAME = f"{PROJECT_ID}-auracart-ml-artifacts"
MODEL_NAME = "AuraCart_Segmenter_Champion"
ENDPOINT_NAME = "auracart-prediction-endpoint"
MODEL_ARTIFACT_PATH = "../artifacts/model.joblib"
REQUIREMENTS_FILE = "../artifacts/requirements.txt"

print(f"Deployment system initialized targeting Project: {PROJECT_ID} in {REGION}")


### Environment Requirements Generation

To ensure the Vertex AI pre-built container has the correct dependencies (Scikit-Learn, Pandas, Imbalanced-Learn), we generate a `requirements.txt` file as an deployment artifact. This file will be uploaded to Cloud Storage alongside the model binary.

In [ ]:
import os
# --- Task 4.2.4: Capture Dependencies ---
requirements_content = """
scikit-learn==1.3.0
pandas==2.0.3
numpy==1.24.3
joblib==1.3.1
imbalanced-learn==0.11.0
"""

REQUIREMENTS_FILE = "../artifacts/requirements.txt"
with open(REQUIREMENTS_FILE, "w") as f:
    f.write(requirements_content.strip())

print(f"Deployment requirements manifest generated at: {REQUIREMENTS_FILE}")

### Task 4.2: Model Packaging & Artifact Management

As mandated by the ITS 2140 Specification, **only one model will be deployed** to production: the final champion model trained for the **customer_segment prediction task** (Section 3.3). We will serialize this model into a Scikit-learn Pipeline and prepare the dependency environmental manifest (`requirements.txt`) for cloud containerization.

In [ ]:
print("Task 4.2.5: Uploading Artifacts to Google Cloud Storage (GCS)...")
storage_client = storage.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Success: Bucket {BUCKET_NAME} created.")

# 1. Upload Model Artifact (Requirement 4.2.5 #1)
model_blob = bucket.blob("model/model.joblib")
model_blob.upload_from_filename(MODEL_ARTIFACT_PATH)
print(f"Artifact model.joblib uploaded to gs://{BUCKET_NAME}/model/")

# 2. Upload Requirements Artifact (Requirement 4.2.5 #2)
req_blob = bucket.blob("model/requirements.txt")
req_blob.upload_from_filename(REQUIREMENTS_FILE)
print(f"Artifact requirements.txt uploaded to gs://{BUCKET_NAME}/model/")

### Task 4.3: Deploying a Live Prediction Service with Vertex AI

We now ingest our model into the **Vertex AI Model Registry** and deploy it to a persistent **REST API Endpoint**. This provides a scalable, enterprise-grade inference service for AuraCart's web and mobile frontend applications.

In [ ]:
print("Initializing Vertex AI SDK...")
aiplatform.init(project=PROJECT_ID, location=REGION)

# 1. Upload Model to Registry
print("Registering model in Vertex AI Registry...")
model_registered = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    artifact_uri=f"gs://{BUCKET_NAME}/model/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
)

# 2. Deploy to Endpoint
print("Instantiating Live Prediction Endpoint (this may take several minutes)... ")
endpoint = model_registered.deploy(
    endpoint_display_name=ENDPOINT_NAME,
    machine_type="n1-standard-2", # Minimal compute footprint for operational efficiency
)

print(f"Successfully deployed AuraCart Prediction Engine to Endpoint ID: {endpoint.resource_name}")


### Task 4.3: Deploying a Live Prediction Service with Vertex AI

We now ingest our model into the **Vertex AI Model Registry** and deploy it to a persistent **REST API Endpoint**. This provides a scalable, enterprise-grade inference service for AuraCart's web and mobile frontend applications.

In [ ]:
# --- Task 4.3.4: Test the Live Endpoint via JSON Payload ---
import pandas as pd

# We query using the identical 20-feature input schema defined during Section 3.3 training.
test_instance = [
    # price, quantity, order_month, day, dayofweek, shipping_month, day, dayofweek, delay, is_weekend, popularity,
    # category, delivery_status, payment_method, channel, ship_city, ship_state, bill_city, bill_state, device_type
    155.50, 2, 4, 12, 6, 4, 15, 2, 3, 1, 85, 
    "Home & Kitchen", "Delivered", "Credit Card", "Social Media", "San Jose", "CA", "San Jose", "CA", "Mobile"
]

print("Querying Vertex AI REST API...")
prediction = endpoint.predict(instances=[test_instance])

print(f"\nPrediction Response Object: {prediction}")
print(f"Detected Segment: {prediction.predictions[0]}")
print("--- Task 4.3.4 Satisfied ---")